# 05 - Face Detection & Landmarks

This notebook demonstrates OCI Vision's face detection feature using the
bundled `portrait_demo.png` fixture. Demo mode now returns a real
`FaceDetectionResult`, so the face and landmark workflow works offline.


## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

The `detect_faces()` method invokes OCI Vision's `FACE_DETECTION` feature.
In demo mode we use `portrait_demo.png` to get a full face result with
landmarks.


In [ ]:
result = client.detect_faces("portrait_demo.png")
print(f"Model version: {result.model_version}")
print(f"Faces detected: {len(result.faces)}")

## Explore the results

Each detected face includes a bounding polygon and any available landmarks.
Let's inspect the normalized coordinates first.


In [ ]:
for i, face in enumerate(result.faces, 1):
    print(f"Face {i}: confidence={face.confidence:.1%}")
    print(f"  Position: {face.bounding_polygon.human_position(800, 800)}")
    print(f"  Landmarks: {[lm.type for lm in face.landmarks]}")
    print()

In [ ]:
# Detailed landmark listing
for i, face in enumerate(result.faces, 1):
    print(f"Face {i} landmarks:")
    for lm in face.landmarks:
        px_x = int(lm.x * 800)
        px_y = int(lm.y * 800)
        print(f"  {lm.type:18s} -> normalized ({lm.x:.2f}, {lm.y:.2f}) -> pixels ({px_x}, {px_y})")
    print()

## Visualize

The renderer draws face boxes with blue outlines and marks each landmark as a
blue point.


In [ ]:
from PIL import Image

from oci_vision.core.models import AnalysisReport
from oci_vision.core.renderer import render_overlay
from oci_vision.gallery import get_gallery_path

image_path = get_gallery_path() / "images" / "portrait_demo.png"
base_image = Image.open(image_path)
report = AnalysisReport(image_path=str(image_path), faces=result)
overlay = render_overlay(base_image, report)
overlay

## Under the hood

In [ ]:
import json

raw = result.model_dump()
print(json.dumps(raw, indent=2, default=str))

### Live API usage

```python
client = VisionClient()  # uses ~/.oci/config
result = client.detect_faces("group_photo.jpg")
for face in result.faces:
    print(face.confidence, len(face.landmarks))
```

This is useful for QA overlays, face cropping, and downstream geometry checks.


## Try it yourself

1. Replace the demo portrait with a live group photo.
2. Compute simple geometry from the landmark coordinates.
3. Save `overlay` to disk for a demo slide or report.
4. Combine with `client.analyze(..., features=['faces', 'classification'])`.
